<a href="https://colab.research.google.com/github/chantika-maharani/ProyekAkhir_5C_Estimation-of-Obesity-Levels/blob/main/Notebook/03_varian_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 13 — Tiga Varian Eksperimen

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Berkas yang dibutuhkan | `data/data_bersih.csv` dari Pertemuan 12 |
| Luaran wajib | tiga berkas `varianN.json`, tiga gambar, `tabel_perbandingan.md` |
| Dinilai pada | Progres 2, Pertemuan 14 — **35% perorangan** + **10% kelompok** |
| Tenggat | seluruhnya selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Anggota 1 (Varian 1) | _(isi nama dan NIM)_ |
| Anggota 2 (Varian 2) | _(isi nama dan NIM)_ |
| Anggota 3 (Varian 3) | _(isi nama dan NIM)_ |

---

## Cara memakai notebook ini

Notebook ini memuat **empat bagian**, tetapi Anda **tidak mengerjakan semuanya**.

| Bagian | Dikerjakan oleh | Sel |
|---|---|---|
| Persiapan | semua orang, setiap kali membuka notebook | 0 |
| **Bagian 2** — Varian 1, Metode Elbow | **Anggota 1** saja | 2.1 – 2.4 |
| **Bagian 3** — Varian 2, Silhouette Score | **Anggota 2** saja | 3.1 – 3.4 |
| **Bagian 4** — Varian 3, Pemilihan Fitur | **Anggota 3** saja | 4.1 – 4.4 |
| **Bagian 5** — Tabel Perbandingan | kelompok, setelah ketiga varian selesai | 5.1 – 5.4 |

Nomor Bagian selalu sama dengan digit pertama nomor selnya, jadi Anggota 2 cukup mengingat: **semua sel yang berawalan 3**.

**Langkah untuk tiap anggota:**

1. **File → Save a copy in Drive.** Masing-masing anggota menyimpan salinannya sendiri — jangan bertiga mengerjakan satu salinan.
2. Jalankan **Bagian 0 Persiapan**.
3. Jalankan **hanya bagian varian Anda sendiri**. Jangan menjalankan bagian milik anggota lain: riwayat commit dipakai sebagai bukti penilaian perorangan.
4. Setelah selesai, unduh notebook Anda lewat *File → Download → Download .ipynb*, beri nama `03_varian_1.ipynb`, `03_varian_2.ipynb`, atau `03_varian_3.ipynb` sesuai varian Anda, lalu commit ke folder `notebook/`.
5. Bagian 4 dikerjakan bersama setelah ketiga varian selesai, dan disimpan sebagai `04_perbandingan.ipynb`.

**Urutannya tidak dapat dibalik.** Varian 2 membutuhkan `varian1.json`, dan Varian 3 membutuhkan `varian2.json`. Jadi Anggota 1 harus selesai lebih dulu, lalu Anggota 2, lalu Anggota 3.

Panduan lengkapnya ada di dokumen **Manual Pertemuan 13**, yang ikut dalam paket ini.

---

## 0. Persiapan — **jalankan setiap kali notebook dibuka**

Colab menghapus seluruh berkas setiap kali runtime dimulai ulang. Tiga sel di bawah memasang pustaka, menulis `alat.py`, dan menyiapkan dua fungsi pembantu. Ketiganya harus dijalankan lebih dulu, apa pun bagian yang Anda kerjakan di notebook ini.

`alat.py` berisi fungsi yang dipakai ketiga varian. **Jangan mengubahnya sendiri-sendiri:** kalau satu anggota mengubah isi berkas ini, ketiga varian berhenti sebanding dan tabel perbandingan menjadi tidak sah.

In [ ]:
import ast, json, os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

warnings.filterwarnings('ignore')
pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 40)

for d in ('data', 'data/hasil_varian', 'model', 'gambar'):
    os.makedirs(d, exist_ok=True)

SEED = 42
ID_COL = None        # ganti sesuai kolom identitas paket dataset Anda
IQR_PENGALI = 1.5

print('Siap. scikit-learn', sklearn.__version__)
print('Catat versi ini. Angka yang sama nanti ditulis di requirements.txt.')

Siap. scikit-learn 1.6.1
Catat versi ini. Angka yang sama nanti ditulis di requirements.txt.


In [ ]:
%%writefile alat.py
"""alat.py -- fungsi bersama ketiga varian. Jangan diubah sendiri-sendiri."""
from __future__ import annotations

import json
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
RENTANG_K = range(2, 11)
BERKAS_BERSIH = '/content/data_bersih (5).csv'
FOLDER_HASIL = 'data/hasil_varian'

# Label dan satuan untuk antarmuka aplikasi tingkat obesitas (Tahap 3)
LABEL = {
    'FAVC': ('Sering makan makanan tinggi kalori', '0 = tidak, 1 = ya'),
    'FCVC': ('Keseringan makan sayur', 'skala 1-3'),
    'CAEC': ('Ngemil di antara waktu makan', '0 = tidak, 1 = kadang-kadang, 2 = sering, 3 = selalu'),
    'FAF': ('Frekuensi aktivitas fisik', 'skala 0-3'),
    'TUE': ('Waktu memakai perangkat teknologi', 'skala 0-2'),
}


def label_satuan(kolom: str) -> tuple[str, str]:
    return LABEL.get(kolom, (kolom.replace('_', ' ').title(), ''))


def muat_bersih(fitur: list[str] | None = None) -> pd.DataFrame:
    """Membaca data bersih kelompok. Kolom identitas selalu dibuang."""
    df = pd.read_csv(BERKAS_BERSIH)
    # Hanya kolom numerik yang menjadi fitur pembentuk klaster.
    # Kolom identitas tidak membawa informasi kemiripan.
    df = df.select_dtypes(include='number')
    return df[fitur] if fitur else df


def skalakan(df: pd.DataFrame):
    sc = StandardScaler()
    return sc.fit_transform(df), sc


def jalankan_k(X_scaled, rentang=RENTANG_K) -> pd.DataFrame:
    """Menjalankan K-Means untuk tiap k, mengembalikan tabel metrik."""
    baris = []
    for k in rentang:
        km = KMeans(n_clusters=k, init='k-means++', n_init=20,
                    random_state=SEED)
        lab = km.fit_predict(X_scaled)
        baris.append({
            'k': k,
            'inertia': float(km.inertia_),
            'silhouette': float(silhouette_score(X_scaled, lab)),
            'davies_bouldin': float(davies_bouldin_score(X_scaled, lab)),
        })
    return pd.DataFrame(baris)


def titik_siku(ks, inertia) -> int:
    """Titik siku = jarak tegak lurus terjauh dari garis ujung ke ujung."""
    x = np.asarray(ks, dtype=float)
    y = np.asarray(inertia, dtype=float)
    x_n = (x - x.min()) / (x.max() - x.min())
    y_n = (y - y.min()) / (y.max() - y.min())
    x1, y1, x2, y2 = x_n[0], y_n[0], x_n[-1], y_n[-1]
    jarak = np.abs((y2 - y1) * x_n - (x2 - x1) * y_n + x2 * y1 - y2 * x1)
    jarak /= np.hypot(y2 - y1, x2 - x1)
    return int(x[int(np.argmax(jarak))])


def profil_relatif(df: pd.DataFrame, label) -> pd.DataFrame:
    """Simpangan rata-rata tiap klaster terhadap rata-rata seluruh data,
    dalam satuan simpangan baku. Baris = fitur, kolom = klaster."""
    d = df.copy()
    d['cluster'] = label
    rata = d.drop(columns='cluster').mean()
    sd = d.drop(columns='cluster').std().replace(0, np.nan)
    return ((d.groupby('cluster').mean() - rata) / sd).round(2).T


def nama_otomatis(profil: pd.DataFrame) -> dict[str, str]:
    """Dua fitur dengan simpangan terbesar secara mutlak, beserta arahnya.
    WAJIB diganti mahasiswa dengan nama yang bermakna."""
    nama = {}
    for c in profil.columns:
        kol = profil[c]
        kuat = kol.reindex(kol.abs().sort_values(ascending=False).index)[:2]
        nama[str(int(c))] = ', '.join(
            f'{n} {"tinggi" if v > 0 else "rendah"}' for n, v in kuat.items())
    return nama


def simpan_hasil(nomor: int, isi: dict) -> str:
    os.makedirs(FOLDER_HASIL, exist_ok=True)
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    with open(jalur, 'w', encoding='utf-8') as f:
        json.dump(isi, f, indent=2, ensure_ascii=False)
    return jalur


def baca_hasil(nomor: int) -> dict | None:
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    if not os.path.exists(jalur):
        return None
    with open(jalur, encoding='utf-8') as f:
        return json.load(f)


Overwriting alat.py


In [ ]:
def ambil_berkas():
    """Unggah berkas dari komputer, lalu letakkan di folder yang benar."""
    from google.colab import files
    for nama in files.upload():
        if nama.startswith('varian') and nama.endswith('.json'):
            tujuan = 'data/hasil_varian/' + nama
        elif nama.endswith('.pkl') or nama == 'segmen.json':
            tujuan = 'model/' + nama
        elif nama.endswith('.png'):
            tujuan = 'gambar/' + nama
        else:
            tujuan = 'data/' + nama
        shutil.move(nama, tujuan)
        print('->', tujuan)


def perlu(*jalur):
    """Minta unggahan HANYA untuk berkas yang belum ada di sesi ini."""
    kurang = [j for j in jalur if not os.path.exists(j)]
    if not kurang:
        print('Lengkap:', ', '.join(jalur))
        return
    print('Belum ada:', ', '.join(kurang))
    print('Unggah berkas tersebut sekarang.')
    ambil_berkas()
    masih = [j for j in jalur if not os.path.exists(j)]
    if masih:
        raise FileNotFoundError('Masih kurang: ' + ', '.join(masih))
    print('Lengkap.')


print('ambil_berkas() dan perlu() siap dipakai.')

ambil_berkas() dan perlu() siap dipakai.


# Bagian 2 — Varian 1 — Penentuan k dengan Metode Elbow

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 1 |
| Dikerjakan oleh | **Anggota 1** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 |
| Luaran wajib | `data/hasil_varian/varian1.json`, `gambar/varian1_elbow.png` |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(8)_ |
| Paket dataset | _(Estimation of Obecity levels)_ |
| Nama dan NIM | _(Intan Fairuz Nur Asiyan 2488010005)_ |

---

## Panduan Bagian 2

> **Ketentuan Panduan Proyek Akhir, Cabang C, Varian 1:** *Metode Elbow untuk k = 2 sampai 10, lalu tentukan titik sikunya.*

Bagian ini dikerjakan **Anggota 1 saja**, dan dinilai **perorangan**. Jangan menjalankan Bagian milik anggota lain: riwayat commit dipakai sebagai bukti siapa mengerjakan apa.

**Urutan kerja**

1. Jalankan 2.0 (persiapan).
2. Jalankan 2.1. Bila `data_bersih.csv` belum ada di sesi Anda, Colab akan meminta unggahannya.
3. Jalankan 2.2. Perhatikan nilai inertia tiap k dan posisi titik sikunya.
4. **Tulis sendiri** jawaban di 2.3. Interpretasi inilah yang paling banyak membedakan nilai antaranggota.
5. Jalankan 2.4 untuk mengunduh hasil, lalu commit dengan akun GitHub **Anda sendiri**.
6. Kirim `varian1.json` ke Anggota 2. Varian 2 membutuhkannya untuk membandingkan kesimpulan.

**Yang perlu dipahami lebih dulu**

Inertia adalah jumlah kuadrat jarak setiap titik ke pusat klasternya. Nilainya **selalu** turun ketika k dinaikkan, jadi "inertia terkecil" bukan jawaban — kalau k disamakan dengan jumlah baris, inertia menjadi nol dan hasilnya tidak berguna. Yang dicari adalah titik di mana penurunan mulai melandai, karena di situ penambahan klaster berhenti memberi manfaat berarti.

Menunjuk siku dengan mata sering berbeda antarorang. Sel 2.2 memakai cara yang dapat diperiksa ulang: **jarak tegak lurus terjauh** dari garis yang menghubungkan titik k terkecil ke titik k terbesar. Yang dinilai bukan besarnya angka, melainkan apakah titik siku ditentukan dengan cara yang dapat dijelaskan.

Silhouette ikut dihitung di sini, tetapi hanya sebagai metrik pendukung. Dasar Bagian ini adalah inertia.

---

## 2.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan. Yang sudah ada dilewati; yang belum ada akan diminta untuk diunggah. Aman dijalankan berulang kali.

In [ ]:
perlu('data/data_bersih (5).csv')

Lengkap: data/data_bersih (5).csv


## 2.2 Menjalankan varian 1

In [ ]:
import os
# Membuat pintasan agar alat.py dapat menemukan berkas tanpa mengubah kodenya
if not os.path.exists('/content/data_bersih (5).csv') and os.path.exists('/content/data/data_bersih (5).csv'):
    os.symlink('/content/data/data_bersih (5).csv', '/content/data_bersih (5).csv')

import matplotlib
import matplotlib.pyplot as plt

from alat import (jalankan_k, muat_bersih, simpan_hasil, skalakan, titik_siku)

print('=' * 70)
print('VARIAN 1 - PENENTUAN k DENGAN METODE ELBOW')
print('=' * 70)

df = muat_bersih()
X, _ = skalakan(df)
print(f'Data bersih kelompok: {df.shape[0]} baris, {df.shape[1]} fitur\n')

tabel = jalankan_k(X)
tabel['penurunan'] = tabel['inertia'].diff().abs()
tabel['penurunan_persen'] = (tabel['penurunan'] / tabel['inertia'].shift(1) * 100)
print(tabel[['k', 'inertia', 'penurunan', 'penurunan_persen', 'silhouette']]
      .round(3).to_string(index=False))

k_siku = titik_siku(tabel['k'], tabel['inertia'])
sil_siku = float(tabel.loc[tabel['k'] == k_siku, 'silhouette'].iloc[0])
inertia_siku = float(tabel.loc[tabel['k'] == k_siku, 'inertia'].iloc[0])

print(f'\nTitik siku  : k = {k_siku}')
print(f'Inertia     : {inertia_siku:,.2f}')
print(f'Silhouette  : {sil_siku:.4f}  (metrik pendukung, bukan dasar varian ini)')
print('\nCara menentukannya: titik siku diambil pada jarak tegak lurus terjauh')
print('dari garis yang menghubungkan k terkecil dan k terbesar. Cara ini')
print('menggantikan penilaian dengan mata, yang sering berbeda antarorang.')

plt.figure(figsize=(8, 5))
plt.plot(tabel['k'], tabel['inertia'], marker='o')
plt.axvline(k_siku, ls='--', lw=1, color='grey')
plt.annotate(f'titik siku k={k_siku}', xy=(k_siku, inertia_siku),
             xytext=(k_siku + 0.6, inertia_siku * 1.08),
             arrowprops=dict(arrowstyle='->', lw=1))
plt.xlabel('Jumlah klaster (k)'); plt.ylabel('Inertia (WCSS)')
plt.title('Varian 1 - Metode Elbow')
plt.tight_layout(); plt.savefig('gambar/varian1_elbow.png', dpi=110); plt.close()

jalur = simpan_hasil(1, {
    'varian': 1,
    'nama': 'Penentuan k (Metode Elbow)',
    'yang_diubah': 'Jumlah klaster ditentukan dari titik siku kurva inertia',
    'fitur_dipakai': df.columns.tolist(),
    'k_terpilih': k_siku,
    'metrik_utama': {'silhouette': round(sil_siku, 4)},
    'metrik_pendukung': {'inertia': round(inertia_siku, 2)},
    'tabel_k': tabel[['k', 'inertia', 'silhouette', 'davies_bouldin']]
                .round(4).to_dict(orient='records'),
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian1_elbow.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 1 - PENENTUAN k DENGAN METODE ELBOW
Data bersih kelompok: 2087 baris, 5 fitur

 k  inertia  penurunan  penurunan_persen  silhouette
 2 8264.517        NaN               NaN       0.406
 3 6775.816   1488.701            18.013       0.247
 4 5689.302   1086.514            16.035       0.264
 5 4862.243    827.058            14.537       0.281
 6 4266.402    595.841            12.254       0.288
 7 3878.328    388.074             9.096       0.300
 8 3537.897    340.431             8.778       0.317
 9 3277.320    260.577             7.365       0.319
10 3080.860    196.461             5.995       0.308

Titik siku  : k = 5
Inertia     : 4,862.24
Silhouette  : 0.2809  (metrik pendukung, bukan dasar varian ini)

Cara menentukannya: titik siku diambil pada jarak tegak lurus terjauh
dari garis yang menghubungkan k terkecil dan k terbesar. Cara ini
menggantikan penilaian dengan mata, yang sering berbeda antarorang.

Tersimpan: data/hasil_varian/varian1.json  dan  gambar/varian1_elbow.

## 2.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Apa arti titik siku ini dalam konteks kasus kelompok Anda? Jawab dengan menyebut jumlah segmen, bukan hanya angka k.
Jawab : Titik siku pada grafik menunjukkan hasil yang optimal pada k = 5. Hal ini terlihat dari penurunan nilai inertia yang cukup tajam sebelum k = 5, sedangkan setelah k = 5 penurunannya mulai melandai dan tidak terlalu signifikan. Oleh karena itu, k = 5 dipilih sebagai jumlah cluster yang optimal
2. Seberapa tegas sikunya? Bila tumpul, nilai k mana saja yang masuk akal, dan apa pertimbangan memilih salah satunya?
Jawab : Titik siku pada grafik terlihat cukup jelas, meskipun tidak terlalu tajam. Kandidat nilai k yang masuk akal adalah k = 4, k = 5, dan k = 6. Namun, k = 5 dipilih karena merupakan titik perubahan dari penurunan inertia yang cukup tajam menjadi lebih landai. Selain itu, jumlah lima segmen dianggap sebagai pilihan yang seimbang antara ketepatan pengelompokan dan kompleksitas model
3. Apa yang berubah pada aplikasi web bila k Anda yang dipakai?
Jawab : Apabila k = 5 diterapkan pada aplikasi web, sistem akan membagi data menjadi lima segmen atau kelompok. Hasil pengelompokan tersebut akan memengaruhi tampilan informasi segmentasi pada aplikasi, seperti jumlah kelompok, anggota setiap kelompok, karakteristik masing-masing segmen, serta visualisasi hasil clustering. Dengan demikian, aplikasi akan menampilkan hasil segmentasi berdasarkan lima kelompok sesuai dengan model K-Means yang digunakan

## 2.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 2, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian1.json')
files.download('gambar/varian1_elbow.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_1.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian1.json` ke Anggota 2.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_1.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian1.json` ke Anggota 2.


## Checklist Bagian 2

- [ ] `varian1.json` ter-commit di `data/hasil_varian/`
- [ ] `varian1_elbow.png` ter-commit di `gambar/`
- [ ] Notebook ini diunduh, diberi nama `03_varian_1.ipynb`, dan ter-commit di `notebook/`
- [ ] Commit dilakukan dengan akun GitHub Anggota 1 sendiri
- [ ] Ketiga jawaban di 2.3 sudah ditulis
- [ ] `varian1.json` sudah dikirim ke Anggota 2

# Bagian 3 — Varian 2 — Validasi k dengan Silhouette Score

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 2 |
| Dikerjakan oleh | **Anggota 2** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 dan `varian1.json` dari Anggota 1 |
| Luaran wajib | `data/hasil_varian/varian2.json`, `gambar/varian2_silhouette.png` |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _Tiara Salsabila 2488010045_ |

---

## Panduan Bagian 3

> **Ketentuan Panduan Proyek Akhir, Cabang C, Varian 2:** *Silhouette Score untuk tiap k. Jika kesimpulannya berbeda dari Elbow, jelaskan mengapa.*

Bagian ini dikerjakan **Anggota 2 saja**, dan dinilai **perorangan**.

**Urutan kerja**

1. Jalankan 3.0 (persiapan).
2. Jalankan 3.1. Anda membutuhkan dua berkas: `data_bersih.csv` dan `varian1.json` dari Anggota 1. Minta berkasnya lebih dulu kalau belum ada.
3. Jalankan 3.2.
4. **Tulis sendiri** jawaban di 3.3.
5. Jalankan 3.4, commit dengan akun Anda sendiri, lalu kirim `varian2.json` ke Anggota 3.

**Yang perlu dipahami lebih dulu**

Silhouette Score mengukur, untuk setiap titik, seberapa dekat ia ke klasternya sendiri dibanding ke klaster terdekat berikutnya. Nilainya antara −1 dan 1. Berbeda dari inertia, nilainya **tidak** otomatis membaik ketika k dinaikkan, jadi nilai tertinggi memang dapat dipakai sebagai pertimbangan.

Patokan tafsiran yang dipakai mata kuliah ini:

| Nilai | Tafsiran |
|---|---|
| di atas 0,50 | pemisahan klaster jelas |
| 0,25 sampai 0,50 | struktur lemah, tetapi masih dapat dipertanggungjawabkan |
| di bawah 0,25 | struktur hampir tidak ada |

Pada data pelanggan nyata, angka 0,20 sampai 0,35 adalah hal biasa. Angka rendah bukan berarti pekerjaan Anda salah; yang salah adalah melaporkannya seolah-olah pemisahannya jelas.

**Satu jebakan yang sering terjadi.** Silhouette hampir selalu tertinggi di k=2, karena membagi data menjadi dua bagian besar paling mudah dipisahkan. Nilai itu biasanya tidak dipakai, sebab dua segmen tidak memberi informasi yang berguna untuk tindakan. Kalau k Anda bukan nilai dengan Silhouette tertinggi, **itu justru perlu dijelaskan**, bukan disembunyikan.

Bila kesimpulan Anda berbeda dari Bagian 2, sel di bawah menyusun kerangka penjelasannya secara otomatis — tetapi Anda tetap harus menuliskannya dengan kalimat Anda sendiri.

---

## 3.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan. Yang sudah ada dilewati; yang belum ada akan diminta untuk diunggah. Aman dijalankan berulang kali.

In [67]:
import shutil
import os

# Menyalin berkas data_bersih yang ada ke nama target tanpa tanda nomor (5)
for source in ['data/data_bersih (5).csv', 'data/data_bersih (5).csv', '/content/data_bersih (5).csv']:
    if os.path.exists(source):
        shutil.copy(source, 'data/data_bersih.csv')
        break

perlu('data/data_bersih.csv', 'data/hasil_varian/varian1.json')

Lengkap: data/data_bersih.csv, data/hasil_varian/varian1.json


## 3.2 Menjalankan varian 2

In [73]:
import os
import shutil

# Memastikan berkas data_bersih (5).csv tersedia di path yang dicari oleh alat.py
if not os.path.exists('/content/data_bersih (5).csv'):
    if os.path.exists('data/data_bersih.csv'):
        shutil.copy('data/data_bersih.csv', '/content/data_bersih (5).csv')
    elif os.path.exists('data/data_bersih (5).csv'):
        shutil.copy('data/data_bersih (5).csv', '/content/data_bersih (5).csv')

import matplotlib
import matplotlib.pyplot as plt

from alat import baca_hasil, jalankan_k, muat_bersih, simpan_hasil, skalakan

print('=' * 70)
print('VARIAN 2 - VALIDASI k DENGAN SILHOUETTE SCORE')
print('=' * 70)

df = muat_bersih()
X, _ = skalakan(df)
print(f'Data bersih kelompok: {df.shape[0]} baris, {df.shape[1]} fitur\n')

tabel = jalankan_k(X)
print(tabel[['k', 'silhouette', 'davies_bouldin', 'inertia']]
      .round(4).to_string(index=False))

k_terbaik = int(tabel.loc[tabel['silhouette'].idxmax(), 'k'])
sil_terbaik = float(tabel['silhouette'].max())

print(f'\nSilhouette tertinggi : k = {k_terbaik}  ({sil_terbaik:.4f})')
if sil_terbaik >= 0.5:
    tafsir = 'pemisahan klaster jelas'
elif sil_terbaik >= 0.25:
    tafsir = 'struktur lemah tetapi masih dapat dipertanggungjawabkan'
else:
    tafsir = 'struktur klaster hampir tidak ada, pilihan fitur perlu ditinjau'
print(f'Tafsiran nilainya    : {tafsir}')

# --- Bandingkan dengan kesimpulan Varian 1 ---
v1 = baca_hasil(1)
print('\n' + '-' * 70)
if v1 is None:
    print('Hasil Varian 1 belum ada. Jalankan varian1_elbow.py lebih dulu')
    print('agar perbandingan antarvarian dapat disusun.')
    k_elbow, beda, penjelasan = None, None, None
else:
    k_elbow = v1['k_terpilih']
    beda = k_elbow != k_terbaik
    print(f'Kesimpulan Varian 1 (Elbow)      : k = {k_elbow}')
    print(f'Kesimpulan Varian 2 (Silhouette) : k = {k_terbaik}')
    if not beda:
        penjelasan = ('Kedua varian menunjuk nilai k yang sama, sehingga '
                      'pilihan k berdiri di atas dua dasar yang saling '
                      'menguatkan.')
    else:
        sil_elbow = float(tabel.loc[tabel['k'] == k_elbow, 'silhouette'].iloc[0])
        selisih = sil_terbaik - sil_elbow
        penjelasan = (
            f'Kedua varian berbeda. Elbow mengukur seberapa rapat anggota '
            f'terhadap pusat klasternya sendiri, sedangkan Silhouette juga '
            f'memperhitungkan jarak ke klaster tetangga. Inertia selalu turun '
            f'saat k bertambah, sehingga titik sikunya bergantung pada bentuk '
            f'kurva, bukan pada mutu pemisahan. Pada k={k_elbow} Silhouette '
            f'bernilai {sil_elbow:.4f}, selisih {selisih:.4f} dari nilai '
            f'tertinggi di k={k_terbaik}.')
        if k_terbaik == 2:
            penjelasan += (' Perlu dicatat, Silhouette sering memuncak di k=2 '
                           'karena dua klaster paling mudah dipisahkan, '
                           'padahal dua segmen jarang berguna untuk '
                           'pengambilan keputusan.')
    print('\n' + penjelasan)
print('-' * 70)

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(tabel['k'], tabel['silhouette'], marker='o', color='green')
plt.axvline(k_terbaik, ls='--', lw=1, color='grey')
if k_elbow:
    plt.axvline(k_elbow, ls=':', lw=1.5, color='tab:blue')
plt.xlabel('k'); plt.ylabel('Silhouette Score'); plt.title('Varian 2 - Silhouette')
plt.subplot(1, 2, 2)
plt.plot(tabel['k'], tabel['davies_bouldin'], marker='o', color='red')
plt.xlabel('k'); plt.ylabel('Davies-Bouldin'); plt.title('Metrik pendukung')
plt.tight_layout(); plt.savefig('gambar/varian2_silhouette.png', dpi=110); plt.close()

jalur = simpan_hasil(2, {
    'varian': 2,
    'nama': 'Validasi k (Silhouette Score)',
    'yang_diubah': 'Jumlah klaster ditentukan dari Silhouette tertinggi',
    'fitur_dipakai': df.columns.tolist(),
    'k_terpilih': k_terbaik,
    'k_menurut_elbow': k_elbow,
    'berbeda_dari_elbow': beda,
    'penjelasan_perbedaan': penjelasan,
    'tafsiran': tafsir,
    'metrik_utama': {'silhouette': round(sil_terbaik, 4)},
    'metrik_pendukung': {
        'inertia': round(float(tabel.loc[tabel['k'] == k_terbaik, 'inertia'].iloc[0]), 2),
        'davies_bouldin': round(float(tabel.loc[tabel['k'] == k_terbaik, 'davies_bouldin'].iloc[0]), 4),
    },
    'tabel_k': tabel[['k', 'inertia', 'silhouette', 'davies_bouldin']]
                .round(4).to_dict(orient='records'),
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian2_silhouette.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 2 - VALIDASI k DENGAN SILHOUETTE SCORE
Data bersih kelompok: 2087 baris, 5 fitur

 k  silhouette  davies_bouldin   inertia
 2      0.4060          1.2397 8264.5166
 3      0.2468          1.5314 6775.8156
 4      0.2636          1.3652 5689.3016
 5      0.2809          1.1676 4862.2432
 6      0.2879          1.0854 4266.4024
 7      0.2999          1.1223 3878.3282
 8      0.3170          1.1038 3537.8970
 9      0.3187          1.0662 3277.3204
10      0.3081          1.1562 3080.8599

Silhouette tertinggi : k = 2  (0.4060)
Tafsiran nilainya    : struktur lemah tetapi masih dapat dipertanggungjawabkan

----------------------------------------------------------------------
Kesimpulan Varian 1 (Elbow)      : k = 5
Kesimpulan Varian 2 (Silhouette) : k = 2

Kedua varian berbeda. Elbow mengukur seberapa rapat anggota terhadap pusat klasternya sendiri, sedangkan Silhouette juga memperhitungkan jarak ke klaster tetangga. Inertia selalu turun saat k bertambah, sehingga titik sikunya b

## 3.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Apakah kesimpulan Anda sama dengan Elbow di Bagian 2? Bila berbeda, jelaskan dengan kalimat Anda sendiri mengapa dua metrik dapat menunjuk k yang berbeda.
2. Bila Silhouette tertinggi ada di k=2, mengapa nilai itu tidak dipakai?
3. Apa arti angka Silhouette ini dalam konteks kasus kelompok Anda? Pakai patokan tafsiran di atas, jangan hanya menyebut angkanya.

_(jawab di sini — klik dua kali pada sel untuk menyuntingnya)_

Jawab: Berbeda. Metode Elbow pada Varian 1 menyarankan k=5 berdasarkan titik siku perubahan inertia, sedangkan Varian 2 (Silhouette Score) menyarankan k=2 sebagai nilai tertinggi. Perbedaan ini terjadi karena Elbow (Inertia) hanya mengukur kerapatan internal klaster (seberapa dekat objek ke pusat klasternya sendiri), sehingga nilainya akan selalu turun seiring bertambahnya k . Sementara itu, Silhouette Score mengukur perbandingan jarak internal klaster dengan jarak ke klaster terdekat lainnya (pemisahan), sehingga mampu mendeteksi batas kualitas klaster secara lebih dinamis tanpa selalu menurun ketika k bertambah. Bila Silhouette tertinggi ada di k=2, mengapa nilai itu tidak dipakai?

Jawab: Nilai k=2 secara matematis memberikan pemisahan paling bersih dan mudah karena hanya membagi data menjadi dua kelompok besar. Namun, dalam konteks bisnis dan analisis medis (tingkat obesitas), pembagian menjadi hanya 2 kelompok sangat tidak informatif (misal: hanya mengelompokkan 'Berat Badan Normal' vs 'Berat Badan Berlebih'). Jumlah klaster yang terlalu sedikit ini tidak memberikan segmentasi detail yang berguna untuk melakukan analisis pola kebiasaan atau menentukan penanganan medis yang spesifik. Apa arti angka Silhouette ini dalam konteks kasus kelompok Anda? Pakai patokan tafsiran di atas, jangan hanya menyebut angkanya.

Jawab: Berdasarkan tabel hasil eksperimen, nilai Silhouette untuk k selain 2 berkisar di angka 0.25 hingga 0.30. Berdasarkan patokan tafsiran, nilai ini menunjukkan bahwa struktur klaster pada dataset kita tergolong lemah, tetapi masih dapat dipertanggungjawabkan. Dalam konteks kasus tingkat obesitas kelompok kami, hal ini berarti karakteristik fisik maupun pola hidup sehari-hari antar pasien memiliki kemiripan atau tumpang tindih (overlap) yang cukup tinggi, sehingga batas pemisah antar-kategori klaster tidak terlalu kontras atau tegas.

## 3.4 Unduh, commit, dan teruskan

In [74]:
# Unduh hasil Bagian 3, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian2.json')
files.download('gambar/varian2_silhouette.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_2.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian2.json` ke Anggota 3.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_2.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian2.json` ke Anggota 3.


## Checklist Bagian 3

- [ ] `varian2.json` ter-commit di `data/hasil_varian/`
- [ ] `varian2_silhouette.png` ter-commit di `gambar/`
- [ ] Notebook ini diunduh, diberi nama `03_varian_2.ipynb`, dan ter-commit di `notebook/`
- [ ] Commit dilakukan dengan akun GitHub Anggota 2 sendiri
- [ ] Perbandingan dengan kesimpulan Elbow sudah ditulis
- [ ] `varian2.json` sudah dikirim ke Anggota 3

# Bagian 4 — Varian 3 — Pemilihan Fitur dan Profil Klaster

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 3 |
| Dikerjakan oleh | **Anggota 3** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 dan `varian2.json` dari Anggota 2 |
| Luaran wajib | `data/hasil_varian/varian3.json`, `gambar/varian3_pemilihan_fitur.png`, tabel profil segmen |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 4

> **Ketentuan Panduan Proyek Akhir, Cabang C, Varian 3:** *Bandingkan hasil klaster dari dua himpunan fitur berbeda, lalu susun profil tiap klaster.*

Bagian ini dikerjakan **Anggota 3 saja**, dan dinilai **perorangan**. Bagian ini juga yang menentukan himpunan fitur untuk model final di Bagian 6, jadi hasilnya dipakai langsung oleh aplikasi web.

**Urutan kerja**

1. Jalankan 4.0 (persiapan).
2. Jalankan 4.1. Anda membutuhkan `data_bersih.csv` dan `varian2.json`.
3. Jalankan 4.2.
4. **Tulis sendiri** jawaban di 4.3, termasuk mengisi tabel nama segmen. Tabel itu dipakai kembali di Bagian 6.
5. Jalankan 4.4 dan commit dengan akun Anda sendiri.

**Yang perlu dipahami lebih dulu**

Bagian ini **tidak mengubah k**. Nilai k diambil dari hasil Bagian 3. Kedua himpunan fitur diuji pada k yang sama, supaya yang dibandingkan benar-benar pengaruh pemilihan fitur, bukan pengaruh jumlah klaster.

Himpunan A adalah seluruh fitur. Himpunan B disusun dengan dua saringan:

1. Dari setiap pasangan fitur dengan korelasi di atas 0,9, salah satunya dibuang — karena bagi K-Means dua fitur kembar berarti satu dimensi mendapat bobot ganda.
2. Fitur yang nilainya hampir sama di semua klaster dibuang — karena fitur seperti itu tidak membantu membedakan segmen, tetapi tetap menambah jarak.

Setelah salah satu himpunan terpilih, bagian kedua ketentuan dikerjakan: **menyusun profil tiap klaster**. Profil dibaca dari nilai rata-rata tiap fitur per klaster, dinyatakan relatif terhadap rata-rata keseluruhan.

**Aturan penting dalam memberi nama segmen.** Nama harus dapat ditelusuri ke kolom datanya. "Pengguna saldo besar dengan pembayaran minimum" sah, karena saldo dan pembayaran minimum adalah kolom yang ada. "Pelanggan loyal" tidak sah, karena tidak ada kolom kesetiaan di dalam dataset — nama itu menyimpulkan hal yang tidak diukur.

---

## 4.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan. Yang sudah ada dilewati; yang belum ada akan diminta untuk diunggah. Aman dijalankan berulang kali.

In [ ]:
perlu('data/data_bersih.csv', 'data/hasil_varian/varian2.json')

Lengkap: data/data_bersih.csv, data/hasil_varian/varian2.json


## 4.2 Menjalankan varian 3

In [ ]:
from __future__ import annotations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score

from alat import (SEED, baca_hasil, muat_bersih, nama_otomatis, profil_relatif,
                  simpan_hasil, skalakan)

class args:
    k = 0                   # 0 berarti ikut hasil Bagian 3
    ambang_korelasi = 0.9   # pasangan fitur di atas ini dianggap kembar
    ambang_beda = 0.25      # fitur di bawah ini dianggap tidak membedakan

print('=' * 70)
print('VARIAN 3 - PEMILIHAN FITUR DAN PROFIL KLASTER')
print('=' * 70)

df = muat_bersih()
semua = df.columns.tolist()

v2 = baca_hasil(2)
if args.k:
    K, asal_k = args.k, 'ditetapkan lewat argumen --k'
elif v2:
    K, asal_k = v2['k_terpilih'], 'mengikuti hasil Varian 2'
else:
    K, asal_k = 4, 'nilai bawaan, Varian 2 belum dijalankan'
print(f'Jumlah klaster dipakai: k = {K}  ({asal_k})')
print('Kedua himpunan fitur diuji pada k yang SAMA, agar yang dibandingkan')
print('benar-benar pengaruh pemilihan fitur, bukan pengaruh perbedaan k.\n')


def uji(fitur: list[str]) -> dict:
    sub = df[fitur]
    X, _ = skalakan(sub)
    km = KMeans(n_clusters=K, init='k-means++', n_init=20, random_state=SEED)
    lab = km.fit_predict(X)
    return {
        'fitur': fitur,
        'jumlah_fitur': len(fitur),
        'silhouette': float(silhouette_score(X, lab)),
        'davies_bouldin': float(davies_bouldin_score(X, lab)),
        'inertia': float(km.inertia_),
        'label': lab,
        'X': X,
        'sub': sub,
    }


# ---------- Himpunan A: seluruh fitur ----------
A = uji(semua)

# ---------- Menyusun Himpunan B dengan dua saringan ----------
print('-' * 70)
print('Saringan 1 - membuang salah satu dari pasangan fitur yang kembar')
kor = df.corr().abs()
buang_kembar = set()
for i, a in enumerate(semua):
    for b in semua[i + 1:]:
        if a in buang_kembar or b in buang_kembar:
            continue
        if kor.loc[a, b] > args.ambang_korelasi:
            # buang yang rata-rata korelasinya ke fitur lain lebih tinggi
            korban = a if kor[a].mean() >= kor[b].mean() else b
            buang_kembar.add(korban)
            print(f'  {a} vs {b}: r={kor.loc[a, b]:.3f} -> buang {korban}')
if not buang_kembar:
    print(f'  tidak ada pasangan dengan korelasi di atas {args.ambang_korelasi}')

print('')
print('Saringan 2 - membuang fitur yang nilainya hampir sama di semua klaster')
profil_A = profil_relatif(A['sub'], A['label'])
beda_maks = profil_A.abs().max(axis=1).sort_values()
buang_datar = [f for f, v in beda_maks.items() if v < args.ambang_beda]
for f in buang_datar:
    print(f'  {f:<34} simpangan maksimum {beda_maks[f]:.2f} sd -> dibuang')
if not buang_datar:
    print(f'  tidak ada fitur dengan simpangan di bawah {args.ambang_beda} sd')

fitur_B = [f for f in semua if f not in buang_kembar and f not in buang_datar]
if len(fitur_B) < 3:
    print('\n  Saringan menyisakan terlalu sedikit fitur. '
          'Himpunan B dikembalikan ke seluruh fitur kecuali yang kembar.')
    fitur_B = [f for f in semua if f not in buang_kembar]

B = uji(fitur_B)

# ---------- Perbandingan ----------
print('\n' + '=' * 70)
print('PERBANDINGAN DUA HIMPUNAN FITUR')
print('=' * 70)
print(f'{"":<24}{"Himpunan A":>14}{"Himpunan B":>14}')
print(f'{"Jumlah fitur":<24}{A["jumlah_fitur"]:>14}{B["jumlah_fitur"]:>14}')
print(f'{"Silhouette (utama)":<24}{A["silhouette"]:>14.4f}{B["silhouette"]:>14.4f}')
print(f'{"Davies-Bouldin":<24}{A["davies_bouldin"]:>14.4f}{B["davies_bouldin"]:>14.4f}')
print(f'{"Inertia":<24}{A["inertia"]:>14.2f}{B["inertia"]:>14.2f}')

selisih = B['silhouette'] - A['silhouette']
terbaik, nama_terbaik = (B, 'B') if selisih > 0 else (A, 'A')
print(f'\nSelisih Silhouette B dikurangi A: {selisih:+.4f}')
print(f'Himpunan terpilih: {nama_terbaik} ({terbaik["jumlah_fitur"]} fitur)')

if abs(selisih) < 0.01:
    catatan = ('Selisihnya di bawah 0,01 sehingga kedua himpunan praktis '
               'setara. Himpunan dengan fitur lebih sedikit tetap lebih '
               'disukai karena aplikasi webnya meminta lebih sedikit isian '
               'kepada pengguna.')
    terbaik, nama_terbaik = B, 'B'
elif selisih > 0:
    catatan = ('Membuang fitur yang tidak membedakan klaster menaikkan mutu '
               'pemisahan. Fitur semacam itu hanya menambah dimensi tanpa '
               'menambah informasi, sehingga melemahkan perhitungan jarak.')
else:
    catatan = ('Himpunan lengkap justru lebih baik. Fitur yang disaring '
               'ternyata masih menyumbang pembeda, sehingga penyaringan '
               'tidak diteruskan.')
print(f'Catatan: {catatan}')

# ---------- Profil tiap klaster pada himpunan terpilih ----------
print('\n' + '=' * 70)
print(f'PROFIL TIAP KLASTER - HIMPUNAN {nama_terbaik}')
print('=' * 70)
profil = profil_relatif(terbaik['sub'], terbaik['label'])
print('Satuan: simpangan baku terhadap rata-rata seluruh data.')
print('Nilai di atas +0,5 jauh lebih tinggi, di bawah -0,5 jauh lebih rendah.\n')
print(profil.to_string())

jumlah = {str(int(c)): int((terbaik['label'] == c).sum())
          for c in np.unique(terbaik['label'])}
print('\nJumlah anggota:')
for c, n in jumlah.items():
    print(f'  Klaster {c}: {n:>5} baris ({n / len(df) * 100:5.1f}%)')

nama = nama_otomatis(profil)
print('\nUsulan nama otomatis - WAJIB diganti dengan nama yang bermakna:')
for c, nm in nama.items():
    print(f'  Klaster {c}: {nm}')

# ---------- Gambar ----------
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(['A\nlengkap', 'B\ndisaring'], [A['silhouette'], B['silhouette']],
          color=['tab:blue', 'tab:green'])
ax[0].set_ylabel('Silhouette Score')
ax[0].set_title(f'Varian 3 - pengaruh pemilihan fitur (k={K})')
for i, v in enumerate([A['silhouette'], B['silhouette']]):
    ax[0].text(i, v, f'{v:.4f}', ha='center', va='bottom')

koor = PCA(n_components=2, random_state=SEED).fit_transform(terbaik['X'])
for c in np.unique(terbaik['label']):
    m = terbaik['label'] == c
    ax[1].scatter(koor[m, 0], koor[m, 1], s=12, alpha=0.6, label=f'Klaster {c}')
ax[1].set_title(f'Sebaran klaster, himpunan {nama_terbaik}')
ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig('gambar/varian3_pemilihan_fitur.png', dpi=110)
plt.close()

jalur = simpan_hasil(3, {
    'varian': 3,
    'nama': 'Pemilihan fitur dan profil klaster',
    'yang_diubah': f'Himpunan fitur: A {A["jumlah_fitur"]} fitur '
                   f'dibandingkan B {B["jumlah_fitur"]} fitur, pada k={K}',
    'k_dipakai': K,
    'asal_k': asal_k,
    'himpunan_A': {'fitur': A['fitur'], 'silhouette': round(A['silhouette'], 4),
                   'davies_bouldin': round(A['davies_bouldin'], 4),
                   'inertia': round(A['inertia'], 2)},
    'himpunan_B': {'fitur': B['fitur'], 'silhouette': round(B['silhouette'], 4),
                   'davies_bouldin': round(B['davies_bouldin'], 4),
                   'inertia': round(B['inertia'], 2)},
    'dibuang_kembar': sorted(buang_kembar),
    'dibuang_datar': buang_datar,
    'himpunan_terpilih': nama_terbaik,
    'fitur_terpilih': terbaik['fitur'],
    'k_terpilih': K,
    'metrik_utama': {'silhouette': round(terbaik['silhouette'], 4)},
    'metrik_pendukung': {'inertia': round(terbaik['inertia'], 2),
                         'davies_bouldin': round(terbaik['davies_bouldin'], 4)},
    'catatan': catatan,
    'jumlah_anggota': jumlah,
    'profil_relatif': {str(c): {f: float(v) for f, v in profil[c].items()}
                       for c in profil.columns},
    'nama_otomatis': nama,
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian3_pemilihan_fitur.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 3 - PEMILIHAN FITUR DAN PROFIL KLASTER
Jumlah klaster dipakai: k = 2  (mengikuti hasil Varian 2)
Kedua himpunan fitur diuji pada k yang SAMA, agar yang dibandingkan
benar-benar pengaruh pemilihan fitur, bukan pengaruh perbedaan k.

----------------------------------------------------------------------
Saringan 1 - membuang salah satu dari pasangan fitur yang kembar
  tidak ada pasangan dengan korelasi di atas 0.9

Saringan 2 - membuang fitur yang nilainya hampir sama di semua klaster
  FCVC                               simpangan maksimum 0.07 sd -> dibuang
  TUE                                simpangan maksimum 0.20 sd -> dibuang

PERBANDINGAN DUA HIMPUNAN FITUR
                            Himpunan A    Himpunan B
Jumlah fitur                         5             3
Silhouette (utama)              0.4060        0.5900
Davies-Bouldin                  1.2397        0.8455
Inertia                        8264.52       4102.54

Selisih Silhouette B dikurangi A: +0.1840
Himpunan terp

## 4.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Fitur apa saja yang disaring, dan mengapa pantas disaring? Sebut nama kolomnya dan alasannya satu per satu.
2. Apakah penyaringan menaikkan atau menurunkan Silhouette? Apa artinya bagi kelompok Anda?
3. **Beri nama tiap segmen** berdasarkan angka profil di atas, dengan aturan nama yang dapat ditelusuri ke kolomnya.

Isi tabel berikut. Tabel ini dipakai lagi di Bagian 6 dan muncul di aplikasi web, jadi isi dengan serius.

| Klaster | Jumlah anggota | Fitur menonjol tinggi | Fitur menonjol rendah | Nama segmen | Tindakan yang disarankan |
|---|---|---|---|---|---|
| 0 | | | | | |
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

_(jawab di sini — klik dua kali pada sel untuk menyuntingnya)_

## 4.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 4, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian3.json')
files.download('gambar/varian3_pemilihan_fitur.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_3.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian3.json` ke kelompok untuk Bagian 5.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_3.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian3.json` ke kelompok untuk Bagian 5.


## Checklist Bagian 4

- [ ] `varian3.json` ter-commit di `data/hasil_varian/`
- [ ] `varian3_pemilihan_fitur.png` ter-commit di `gambar/`
- [ ] Notebook ini diunduh, diberi nama `03_varian_3.ipynb`, dan ter-commit di `notebook/`
- [ ] Commit dilakukan dengan akun GitHub Anggota 3 sendiri
- [ ] Tabel nama segmen sudah terisi lengkap, bukan nama otomatis
- [ ] Setiap nama segmen dapat ditelusuri ke kolom datanya
- [ ] `varian3.json` sudah diteruskan ke kelompok

# Bagian 5 — Tabel Perbandingan Tiga Varian

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | Pertemuan 14 (Progres 2) |
| Tahap resmi | Tahap 2 — Perbandingan dan Pemilihan Model |
| Dikerjakan oleh | kelompok |
| Berkas yang dibutuhkan | ketiga berkas `varian1.json`, `varian2.json`, `varian3.json` |
| Luaran wajib | `data/tabel_perbandingan.md` dan satu paragraf kesimpulan |
| Dinilai pada | Progres 2, Pertemuan 14, **10%** (nilai kelompok) |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | _(isi)_ |
| Paket dataset | _(isi sesuai penetapan dosen)_ |
| Nama dan NIM | _(isi)_ |

---

## Panduan Bagian 5

Tabel ini adalah bukti utama bahwa kelompok benar-benar bereksperimen, bukan menjalankan satu kode lalu menyebutnya tiga varian. Karena itu tabelnya harus dibangun dari **ketiga berkas json** yang dihasilkan masing-masing anggota, bukan ditulis ulang dengan tangan.

**Urutan kerja**

1. Pastikan ketiga anggota sudah menyelesaikan Bagian 2, 3, dan 4 serta sudah commit berkas json-nya.
2. Jalankan 5.0 (persiapan).
3. Jalankan 5.1. Bila ada json yang belum ada, Colab akan memintanya.
4. Jalankan 5.2 untuk menyusun tabel.
5. **Tulis sendiri** paragraf kesimpulan di 5.3. Paragraf ini menentukan varian mana yang dipakai aplikasi web.
6. Tempelkan tabel dan paragraf kesimpulan ke `README.md`, lalu commit.

**Yang perlu dipahami lebih dulu**

Ketiga varian **tidak dinilai dari siapa yang angkanya paling bagus**. Varian 1 dan 2 menjawab pertanyaan berapa k, Varian 3 menjawab pertanyaan fitur mana. Jadi tabel ini bukan lomba, melainkan dasar untuk satu keputusan: himpunan fitur dan nilai k mana yang dipakai model final.

Tiga metrik yang dilaporkan:

| Metrik | Arah yang lebih baik | Yang diukur |
|---|---|---|
| Silhouette Score | makin besar | seberapa terpisah antarklaster |
| Davies-Bouldin Index | makin kecil | rasio sebaran dalam klaster terhadap jarak antarklaster |
| Inertia | makin kecil, tetapi hanya pada k yang sama | jumlah kuadrat jarak ke pusat klaster |

Inertia **tidak boleh** dibandingkan antar-k yang berbeda, karena nilainya selalu turun saat k naik. Kesalahan ini sering muncul pada tabel perbandingan mahasiswa.

---

## 5.1 Mengambil berkas yang diperlukan

Kumpulkan ketiga berkas json dari anggota lain lebih dulu. Kalau salah satu belum ada, Bagian ini tidak dapat diselesaikan — dan itu berarti ada anggota yang belum menyelesaikan variannya.

In [ ]:
perlu('data/data_bersih.csv', 'data/hasil_varian/varian1.json',
      'data/hasil_varian/varian2.json', 'data/hasil_varian/varian3.json')

Lengkap: data/data_bersih.csv, data/hasil_varian/varian1.json, data/hasil_varian/varian2.json, data/hasil_varian/varian3.json


## 5.2 Menyusun tabel

In [ ]:
from __future__ import annotations

import pandas as pd

from alat import baca_hasil

print('=' * 70)
print('TABEL PERBANDINGAN TIGA VARIAN')
print('=' * 70)

hasil = {n: baca_hasil(n) for n in (1, 2, 3)}
hilang = [n for n, h in hasil.items() if h is None]
if hilang:
    print('Varian berikut belum dijalankan: ' +
          ', '.join(str(n) for n in hilang))
    print('Jalankan skripnya lebih dulu, lalu ulangi perintah ini.')
    raise SystemExit(1)

baris = []
for n in (1, 2, 3):
    h = hasil[n]
    pendukung = ', '.join(f'{k.replace("_", " ")} {v}'
                          for k, v in h['metrik_pendukung'].items())
    if n == 1:
        catatan = f'Titik siku pada k={h["k_terpilih"]}'
    elif n == 2:
        catatan = ('Sama dengan kesimpulan Elbow'
                   if h.get('berbeda_dari_elbow') is False
                   else f'Berbeda dari Elbow (k={h.get("k_menurut_elbow")})')
    else:
        catatan = (f'Himpunan {h["himpunan_terpilih"]} terpilih, '
                   f'{len(h["fitur_terpilih"])} fitur')
    baris.append({
        'Varian': n,
        'Apa yang diubah': h['yang_diubah'],
        'k': h['k_terpilih'],
        'Silhouette (utama)': h['metrik_utama']['silhouette'],
        'Metrik pendukung': pendukung,
        'Catatan': catatan,
    })

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))

# --- Varian mana yang dipakai untuk aplikasi web ---
terbaik = max((1, 2, 3), key=lambda n: hasil[n]['metrik_utama']['silhouette'])
print(f'\nSilhouette tertinggi ada pada Varian {terbaik} '
      f'({hasil[terbaik]["metrik_utama"]["silhouette"]})')
print('Varian 3 selalu menjadi acuan susunan fitur aplikasi, karena hanya')
print('varian itu yang menguji himpunan fitur. Varian 1 dan 2 menguji k.')

k_final = hasil[3]['k_terpilih']
fitur_final = hasil[3]['fitur_terpilih']
print(f'\nUsulan konfigurasi model final: k={k_final}, '
      f'{len(fitur_final)} fitur dari Varian 3.')

# --- Berkas markdown untuk README ---
md = ['## Tabel perbandingan tiga varian', '',
      '| Varian | Apa yang diubah | k | Silhouette (utama) | Metrik pendukung | Catatan |',
      '| --- | --- | --- | --- | --- | --- |']
for r in baris:
    md.append(f'| {r["Varian"]} | {r["Apa yang diubah"]} | {r["k"]} | '
              f'{r["Silhouette (utama)"]} | {r["Metrik pendukung"]} | '
              f'{r["Catatan"]} |')
md += ['', '### Kesimpulan', '',
       '_Paragraf ini WAJIB ditulis sendiri oleh kelompok dan menjawab tiga hal:_',
       '',
       f'1. Varian mana yang dipakai untuk aplikasi web. _(usulan: Varian 3, '
       f'k={k_final}, {len(fitur_final)} fitur)_',
       '2. Mengapa varian itu dipilih.',
       '3. Apa yang dikorbankan dengan memilihnya.',
       '',
       '_Jawaban berupa varian dengan angka tertinggi, tanpa alasan lain, '
       'belum memadai._']

with open('data/tabel_perbandingan.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(md))
tabel.to_csv('data/tabel_perbandingan.csv', index=False)

print('\nTersimpan: data/tabel_perbandingan.md (tempel ke README.md)')
print('           data/tabel_perbandingan.csv')

TABEL PERBANDINGAN TIGA VARIAN
 Varian                                            Apa yang diubah  k  Silhouette (utama)                       Metrik pendukung                      Catatan
      1    Jumlah klaster ditentukan dari titik siku kurva inertia  5              0.2809                        inertia 4862.24          Titik siku pada k=5
      2        Jumlah klaster ditentukan dari Silhouette tertinggi  2              0.4060 inertia 8264.52, davies bouldin 1.2397     Berbeda dari Elbow (k=5)
      3 Himpunan fitur: A 5 fitur dibandingkan B 3 fitur, pada k=2  2              0.5900 inertia 4102.54, davies bouldin 0.8455 Himpunan B terpilih, 3 fitur

Silhouette tertinggi ada pada Varian 3 (0.59)
Varian 3 selalu menjadi acuan susunan fitur aplikasi, karena hanya
varian itu yang menguji himpunan fitur. Varian 1 dan 2 menguji k.

Usulan konfigurasi model final: k=2, 3 fitur dari Varian 3.

Tersimpan: data/tabel_perbandingan.md (tempel ke README.md)
           data/tabel_perbandingan.

## 5.3 Paragraf kesimpulan, wajib ditulis sendiri

Paragraf ini menjawab tiga hal:

1. **Varian mana yang dipakai untuk aplikasi web?** Sebut nilai k dan himpunan fiturnya.
2. **Mengapa varian itu dipilih?** Sebut metrik dan pertimbangan praktisnya.
3. **Apa yang dikorbankan dengan memilihnya?** Setiap pilihan punya konsekuensi.

Jawaban berupa "varian 3 dipilih karena Silhouette-nya paling tinggi", tanpa alasan lain, **belum memadai**. Nilai tertinggi saja bukan alasan: harus dijelaskan mengapa angka itu berarti dalam konteks kasus kelompok Anda, dan apa yang hilang dari pilihan itu.

_(tulis di sini — klik dua kali pada sel untuk menyuntingnya)_

## 5.4 Unduh dan commit

In [ ]:
from google.colab import files
files.download('data/tabel_perbandingan.md')
print()
print('Langkah berikutnya:')
print('  1. Tempelkan isi tabel_perbandingan.md ke README.md.')
print('  2. Tambahkan paragraf kesimpulan 5.3 di bawah tabel itu.')
print('  3. Commit keduanya ke repositori kelompok.')
print('  4. Unduh notebook ini sebagai 04_perbandingan.ipynb, commit ke notebook/.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Tempelkan isi tabel_perbandingan.md ke README.md.
  2. Tambahkan paragraf kesimpulan 5.3 di bawah tabel itu.
  3. Commit keduanya ke repositori kelompok.
  4. Unduh notebook ini sebagai 04_perbandingan.ipynb, commit ke notebook/.


## Checklist Bagian 5

- [ ] Tabel dibangun dari ketiga berkas json, bukan ditulis ulang dengan tangan
- [ ] `data/tabel_perbandingan.md` ter-commit
- [ ] Tabel sudah ditempel di `README.md`
- [ ] Paragraf kesimpulan sudah ditulis dan menjawab ketiga hal di atas
- [ ] Inertia tidak dibandingkan antar-k yang berbeda
- [ ] Nilai k dan himpunan fitur untuk model final sudah ditetapkan